In [1]:
# =============================================================================
# P653 - Diabetic Retinopathy Prediction
# Binary Classification Project
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, roc_curve
)



In [ ]:
# =============================================================================
# 1. LOAD DATA
# =============================================================================

df = pd.read_csv('P653_pronostico_dataset.csv', sep=';')

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nFirst 5 rows:\n{df.head()}")



In [ ]:
# =============================================================================
# 2. EXPLORATORY DATA ANALYSIS (EDA)
# =============================================================================

print("\n" + "=" * 60)
print("EXPLORATORY DATA ANALYSIS")
print("=" * 60)

print(f"\nDescriptive Statistics:\n{df.describe()}")
print(f"\nMissing Values:\n{df.isnull().sum()}")
print(f"\nTarget Distribution:\n{df['prognosis'].value_counts()}")
print(f"\nTarget Distribution (%):\n{df['prognosis'].value_counts(normalize=True) * 100}")

# --- EDA Plots ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('EDA - Diabetic Retinopathy Dataset', fontsize=15, fontweight='bold')

# Class Distribution
axes[0, 0].bar(
    ['No Retinopathy', 'Retinopathy'],
    df['prognosis'].value_counts().values,
    color=['#2196F3', '#F44336'], edgecolor='white'
)
axes[0, 0].set_title('Class Distribution')
axes[0, 0].set_ylabel('Count')
for i, v in enumerate(df['prognosis'].value_counts().values):
    axes[0, 0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Feature distributions by class
features = ['age', 'systolic_bp', 'diastolic_bp', 'cholesterol']
plot_positions = [(0, 1), (0, 2), (1, 0), (1, 1)]
for feat, pos in zip(features, plot_positions):
    ax = axes[pos]
    for label, color in [('retinopathy', '#F44336'), ('no_retinopathy', '#2196F3')]:
        ax.hist(df[df['prognosis'] == label][feat], alpha=0.6,
                label=label.replace('_', ' ').title(), bins=30, color=color)
    ax.set_title(f'{feat.replace("_", " ").title()} Distribution')
    ax.set_xlabel(feat)
    ax.legend()

# Correlation Heatmap
df_temp = df[['age', 'systolic_bp', 'diastolic_bp', 'cholesterol']].copy()
df_temp['target'] = (df['prognosis'] == 'retinopathy').astype(int)
corr = df_temp.corr()
im = axes[1, 2].imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
axes[1, 2].set_xticks(range(5))
axes[1, 2].set_yticks(range(5))
labels = ['age', 'sys_bp', 'dia_bp', 'chol', 'target']
axes[1, 2].set_xticklabels(labels, fontsize=8)
axes[1, 2].set_yticklabels(labels, fontsize=8)
axes[1, 2].set_title('Correlation Matrix')
plt.colorbar(im, ax=axes[1, 2])
for i in range(5):
    for j in range(5):
        axes[1, 2].text(j, i, f'{corr.values[i, j]:.2f}',
                        ha='center', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nEDA plots saved as 'eda_plots.png'")

# Boxplots for outlier detection
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('Boxplots - Outlier Detection by Feature', fontsize=13, fontweight='bold')
for ax, feat in zip(axes, features):
    df.boxplot(column=feat, by='prognosis', ax=ax)
    ax.set_title(feat.replace('_', ' ').title())
    ax.set_xlabel('')
plt.suptitle('Boxplots by Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print("Boxplots saved as 'boxplots.png'")



In [ ]:
# =============================================================================
# 3. FEATURE ENGINEERING & PREPROCESSING
# =============================================================================

print("\n" + "=" * 60)
print("FEATURE ENGINEERING & PREPROCESSING")
print("=" * 60)

# Encode target
le = LabelEncoder()
df['target'] = le.fit_transform(df['prognosis'])
# retinopathy = 1, no_retinopathy = 0
print(f"Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Features and target
X = df[features]
y = df['target']

# Train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain size: {X_train.shape[0]} samples")
print(f"Test size:  {X_test.shape[0]} samples")
print(f"Train class balance: {y_train.value_counts().to_dict()}")
print(f"Test class balance:  {y_test.value_counts().to_dict()}")

# Feature Scaling (for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)



In [ ]:
# =============================================================================
# 4. MODEL BUILDING
# =============================================================================

print("\n" + "=" * 60)
print("MODEL BUILDING & EVALUATION")
print("=" * 60)

models = {
    'Logistic Regression': (LogisticRegression(random_state=42, max_iter=1000), True),
    'Decision Tree':       (DecisionTreeClassifier(random_state=42, max_depth=5), False),
    'Random Forest':       (RandomForestClassifier(random_state=42, n_estimators=100), False),
    'Gradient Boosting':   (GradientBoostingClassifier(random_state=42, n_estimators=100), False),
}

results = {}

for name, (model, needs_scaling) in models.items():
    Xtr = X_train_scaled if needs_scaling else X_train
    Xte = X_test_scaled  if needs_scaling else X_test

    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_prob)
    f1   = f1_score(y_test, y_pred)
    cv   = cross_val_score(model, Xtr, y_train, cv=5, scoring='roc_auc').mean()

    results[name] = {
        'model': model, 'y_pred': y_pred, 'y_prob': y_prob,
        'acc': acc, 'auc': auc, 'f1': f1, 'cv_auc': cv
    }

    print(f"\n--- {name} ---")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  AUC-ROC  : {auc:.4f}")
    print(f"  F1 Score : {f1:.4f}")
    print(f"  CV AUC   : {cv:.4f} (5-fold)")



In [ ]:
# =============================================================================
# 5. MODEL COMPARISON PLOTS
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Model Evaluation Results', fontsize=15, fontweight='bold')

model_names = list(results.keys())
accs  = [results[n]['acc']  for n in model_names]
aucs  = [results[n]['auc']  for n in model_names]
f1s   = [results[n]['f1']   for n in model_names]
x = np.arange(len(model_names))
w = 0.25

axes[0].bar(x - w, accs, w, label='Accuracy',  color='#2196F3')
axes[0].bar(x,      aucs, w, label='AUC-ROC',   color='#4CAF50')
axes[0].bar(x + w, f1s,  w, label='F1 Score',  color='#FF9800')
axes[0].set_xticks(x)
axes[0].set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=9)
axes[0].set_ylim(0.5, 1.05)
axes[0].set_title('Performance Comparison')
axes[0].legend()
axes[0].set_ylabel('Score')

# ROC Curves
for name in results:
    fpr, tpr, _ = roc_curve(y_test, results[name]['y_prob'])
    axes[1].plot(fpr, tpr, label=f"{name} ({results[name]['auc']:.3f})")
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves')
axes[1].legend(fontsize=8)

# Confusion Matrix - Best Model
best = max(results, key=lambda k: results[k]['auc'])
cm = confusion_matrix(y_test, results[best]['y_pred'])
im = axes[2].imshow(cm, cmap='Blues')
axes[2].set_xticks([0, 1])
axes[2].set_yticks([0, 1])
axes[2].set_xticklabels(['No Retinopathy', 'Retinopathy'], fontsize=9)
axes[2].set_yticklabels(['No Retinopathy', 'Retinopathy'], fontsize=9)
axes[2].set_title(f'Confusion Matrix\n({best})')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, cm[i, j], ha='center', va='center',
                     fontsize=14, fontweight='bold',
                     color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.savefig('model_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nModel results saved as 'model_results.png'")

